# Day 39 — Pandas Merging & Joining DataFrames
**Month 3 | Week 2 | Python + Pandas**

> **Real-world framing:** Every real client dataset lives in multiple tables.
> Sales in one sheet, customers in another, products in a third.
> Merging is how you combine them — and it's the most SQL-like thing you'll do in Pandas.
> Get this wrong and your numbers are silently wrong. Get it right and you can answer
> questions no single table can answer.

---

**What you'll learn today:**
- `pd.merge()` — the main tool (inner, left, right, outer joins)
- `pd.concat()` — stacking DataFrames vertically or horizontally
- Handling duplicate columns after a merge
- Diagnosing merge problems (missing rows, unexpected NaNs)

**Skills assumed:** GroupBy (Day 38), Data Cleaning (Day 37), Pandas basics (Day 34)


---
## 📦 Section 1 — Raw Data (Read Only — Do Not Modify)

Three tables from a fictional retail company: **ShopEase**.
You will merge these to answer business questions.


In [20]:
import pandas as pd
import numpy as np

# ── Table 1: Orders ──────────────────────────────────────────────────────────
orders = pd.DataFrame({
    'order_id':    [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    'customer_id': ['C01', 'C02', 'C03', 'C01', 'C04', 'C02', 'C05', 'C03', 'C06', 'C01'],
    'product_id':  ['P01', 'P02', 'P01', 'P03', 'P02', 'P04', 'P01', 'P03', 'P05', 'P02'],
    'quantity':    [2, 1, 3, 1, 2, 4, 1, 2, 1, 3],
    'order_date':  ['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10',
                    '2024-01-12', '2024-01-15', '2024-01-18', '2024-01-20',
                    '2024-01-22', '2024-01-25']
})

# ── Table 2: Customers ───────────────────────────────────────────────────────
customers = pd.DataFrame({
    'customer_id': ['C01', 'C02', 'C03', 'C04', 'C05', 'C07'],  # C06 missing intentionally
    'name':        ['Arjun Sharma', 'Priya Mehta', 'Rahul Verma',
                    'Sneha Iyer', 'Kiran Rao', 'Deepa Nair'],
    'city':        ['Delhi', 'Mumbai', 'Bangalore', 'Chennai', 'Hyderabad', 'Pune'],
    'segment':     ['Premium', 'Standard', 'Standard', 'Premium', 'Standard', 'Premium']
})

# ── Table 3: Products ────────────────────────────────────────────────────────
products = pd.DataFrame({
    'product_id':   ['P01', 'P02', 'P03', 'P04', 'P06'],  # P05 missing intentionally
    'product_name': ['Laptop Stand', 'Wireless Mouse', 'USB Hub', 'Keyboard', 'Monitor'],
    'category':     ['Accessories', 'Accessories', 'Accessories', 'Peripherals', 'Peripherals'],
    'unit_price':   [1200, 850, 650, 1800, 12000]
})

# ── Table 4: Q2 Orders (for concat practice) ─────────────────────────────────
orders_q2 = pd.DataFrame({
    'order_id':    [1011, 1012, 1013],
    'customer_id': ['C02', 'C04', 'C01'],
    'product_id':  ['P01', 'P03', 'P04'],
    'quantity':    [1, 2, 1],
    'order_date':  ['2024-04-03', '2024-04-07', '2024-04-11']
})

print("Orders shape:", orders.shape)
print("Customers shape:", customers.shape)
print("Products shape:", products.shape)
print("Q2 Orders shape:", orders_q2.shape)
print()
print("Note: C06 exists in orders but NOT in customers → used in join exercises")
print("Note: P05 exists in orders but NOT in products → used in join exercises")


Orders shape: (10, 5)
Customers shape: (6, 4)
Products shape: (5, 4)
Q2 Orders shape: (3, 5)

Note: C06 exists in orders but NOT in customers → used in join exercises
Note: P05 exists in orders but NOT in products → used in join exercises


---
## 📖 Section 2 — Concept Notes

### The Four Joins — Mental Model

Think of two circles (like a Venn diagram):

| Join Type | What you get | SQL equivalent |
|-----------|-------------|----------------|
| `inner`   | Only rows that match in BOTH tables | `INNER JOIN` |
| `left`    | All rows from LEFT + matches from right (NaN where no match) | `LEFT JOIN` |
| `right`   | All rows from RIGHT + matches from left (NaN where no match) | `RIGHT JOIN` |
| `outer`   | Everything from both, NaN where no match | `FULL OUTER JOIN` |

### Core Syntax

```python
# Basic merge
pd.merge(left_df, right_df, on='common_column', how='inner')

# Different column names
pd.merge(left_df, right_df, left_on='id', right_on='customer_id', how='left')

# Merge on multiple columns
pd.merge(df1, df2, on=['col1', 'col2'], how='inner')
```

### concat vs merge

| | `pd.merge()` | `pd.concat()` |
|--|---|---|
| Purpose | Combine tables horizontally by matching key | Stack tables vertically (same columns) |
| SQL analogy | JOIN | UNION ALL |
| Key required | Yes (join key) | No |
| Common use | Combine orders + customers | Combine Q1 + Q2 orders |

### The Merge Trap — Duplicate Column Names

When both tables have a column with the same name (but different meaning), Pandas adds suffixes:

```python
# Both tables have 'name' column → result has 'name_x' and 'name_y'
merged = pd.merge(df1, df2, on='id', how='inner')
# Fix: rename BEFORE merge, or use suffixes parameter
merged = pd.merge(df1, df2, on='id', how='inner', suffixes=('_orders', '_customers'))
```

### Diagnosing Merge Problems

```python
# Check for unmatched rows
left_only = merged[merged['right_column'].isna()]   # rows only in left
right_only = merged[merged['left_column'].isna()]   # rows only in right

# Check row count
print("Before merge:", len(df1))
print("After merge:", len(merged))   # Should match your expectation
```


---
## 🏋️ Section 3 — Practice Tasks

**Rules:**
- Use only the 4 DataFrames defined in Section 1
- No hardcoded values — all answers must come from code
- Every task must print output — silent cells score 0
- Follow Number + Reason + Action format for all business insights


### A — Inner Join (20 pts)

**A1 (5 pts):** Merge `orders` and `customers` using an inner join on `customer_id`.
Print the shape and the first 5 rows.
How many orders are in the result vs the original orders table? Print a comparison line.

**A2 (5 pts):** Merge `orders` and `products` using an inner join on `product_id`.
Add a new column `revenue = quantity * unit_price`.
Print the total revenue across all matched orders.

**A3 (10 pts):** Do a 3-table merge: orders → customers → products (both inner joins).
From this merged table:
- Find the top customer by total revenue (name, not ID)
- Find the top-selling category by total revenue
- Print both as: `"Number: X | Reason: Y | Action: Z"`


In [21]:
# A1 — Inner join: orders + customers
a1 = pd.merge(orders, customers, on = 'customer_id', how = 'inner')
print('A1 Inner Join shape:', a1.shape)
print(a1.head(5))
print(f"Original orders: {len(orders)} | After inner join: {len(a1)} | Lost: {len(orders) - len(a1)} (C06 unmatched)")

A1 Inner Join shape: (9, 8)
   order_id customer_id product_id  quantity  order_date          name  \
0      1001         C01        P01         2  2024-01-05  Arjun Sharma   
1      1002         C02        P02         1  2024-01-07   Priya Mehta   
2      1003         C03        P01         3  2024-01-08   Rahul Verma   
3      1004         C01        P03         1  2024-01-10  Arjun Sharma   
4      1005         C04        P02         2  2024-01-12    Sneha Iyer   

        city   segment  
0      Delhi   Premium  
1     Mumbai  Standard  
2  Bangalore  Standard  
3      Delhi   Premium  
4    Chennai   Premium  
Original orders: 10 | After inner join: 9 | Lost: 1 (C06 unmatched)


In [22]:
# A2 — Inner join: orders + products + revenue column
a2 = pd.merge(orders, products, on='product_id', how='inner')
a2['revenue'] = a2['quantity']*a2['unit_price']
print(f"\nA2- Total Revenue(matched revenue only): ₹{a2['revenue'].sum():,.0f}")


A2- Total Revenue(matched revenue only): ₹21,450


In [23]:
# A3 — 3-table merge + top customer + top category
a3 = pd.merge(orders, products, on='product_id', how='inner')
a3 = pd.merge(a3, customers, on='customer_id', how='inner')
a3['revenue'] = a3['quantity']*a3['unit_price']
top_customer = a3.groupby('name')['revenue'].sum().idxmax()
top_customer_rev = a3.groupby('name')['revenue'].sum().max()
top_category = a3.groupby('category')['revenue'].sum().idxmax()
top_category_rev = a3.groupby('category')['revenue'].sum().max()
print(f"\nA3 — Top customer: {top_customer} | Revenue: ₹{top_customer_rev:,.0f}")
print(f"A3 — Top category: {top_category} | Revenue: ₹{top_category_rev:,.0f}")
print(f"Number: {top_customer} generated ₹{top_customer_rev:,.0f} in Q1 | Reason: multiple orders + high-value product mix | Action: offer loyalty reward to retain and upsell.")



A3 — Top customer: Priya Mehta | Revenue: ₹8,050
A3 — Top category: Accessories | Revenue: ₹14,250
Number: Priya Mehta generated ₹8,050 in Q1 | Reason: multiple orders + high-value product mix | Action: offer loyalty reward to retain and upsell.


### B — Left Join (20 pts)

**B1 (5 pts):** Left join `orders` onto `customers` (keep all orders, bring in customer info).
Which order_ids have NaN in the `name` column? Print them.
Why does this happen? Add a 1-line comment explaining.

**B2 (5 pts):** From the B1 result, fill NaN in the `city` column with `"Unknown"`.
Fill NaN in the `segment` column with `"Unregistered"`.
Print the count of orders per segment (including Unregistered).

**B3 (10 pts):** Left join `orders` onto `products`.
Which order_ids have NaN in `product_name`? Print them.
Calculate the total revenue for orders WHERE product info IS available.
Calculate the revenue you are missing due to unmatched products (estimate using average unit_price).
Print a business insight using Number + Reason + Action.


In [24]:
# B1 — Left join: all orders, bring customer info, find NaN rows
b1 = pd.merge(orders, customers, on='customer_id', how='left')
nan_orders = b1[b1['name'].isna()]['order_id'].tolist()
print(f"Orders with - no customer records: {nan_orders}")

Orders with - no customer records: [1009]


In [25]:
# B2 — Fill NaN values + count per segment
b2 = b1.copy()
b2['city'] = b2['city'].fillna('Unknown')
b2['segment'] = b2['segment'].fillna('Unregistered')
print("\nB2 - Orders per segment")
print(b2['segment'].value_counts())


B2 - Orders per segment
segment
Standard        5
Premium         4
Unregistered    1
Name: count, dtype: int64


In [26]:
# B3 — Left join with products + revenue analysis + missing revenue estimate
b3 = pd.merge(orders, products, on='product_id', how='left')
nan_products = b3[b3['product_name'].isna()]['order_id'].tolist()
b3['revenue'] = b3['quantity'] * b3['unit_price']
known_rev = b3[b3['product_name'].notna()]['revenue'].sum()
avg_price = products['unit_price'].mean()
missing_qty = b3[b3['product_name'].isna()]['quantity'].sum()
missing_est = missing_qty * avg_price
print(f"\nB3 — Orders with unknown product: {nan_products}")
print(f"Revenue from matched orders: ₹{known_rev:,.0f}")
print(f"Estimated missing revenue: ₹{missing_est:,.0f}")
print(f"Number: ₹{missing_est:,.0f} estimated revenue untracked | Reason: P05 has no product master record | Action: Add P05 to product table to capture full revenue.")



B3 — Orders with unknown product: [1009]
Revenue from matched orders: ₹21,450
Estimated missing revenue: ₹3,300
Number: ₹3,300 estimated revenue untracked | Reason: P05 has no product master record | Action: Add P05 to product table to capture full revenue.


### C — Outer Join & Merge Diagnostics (20 pts)

**C1 (5 pts):** Do a full outer join between `customers` and `orders` on `customer_id`.
Which customer_ids appear in customers but have NO orders? Print them with their names.
Which customer_ids appear in orders but have NO customer record? Print them.

**C2 (5 pts):** After the C1 outer join, add a column `match_status`:
- `"Both"` — if customer_id appears in both tables
- `"Customer only"` — if the customer has no orders
- `"Order only"` — if the order has no customer record
Print value counts of `match_status`.

**C3 (10 pts):** Merge `orders` and `products` (left join), then merge the result with `customers` (left join).
From this fully merged table, calculate revenue per city.
Print a business insight for the city with highest revenue using Number + Reason + Action.
Flag any city with revenue below 2000 as "Low Revenue Zone".


In [27]:
# C1 — Outer join diagnostics
c1 = pd.merge(customers, orders, on='customer_id', how='outer')
cust_no_orders = c1[c1['order_id'].isna()][['customer_id','name']].drop_duplicates()
order_no_cust  = c1[c1['name'].isna()][['customer_id','order_id']].drop_duplicates()
print(f"\nC1 — Customers with no orders:\n{cust_no_orders}")
print(f"\nC1 — Orders with no customer record:\n{order_no_cust}")


C1 — Customers with no orders:
   customer_id        name
10         C07  Deepa Nair

C1 — Orders with no customer record:
  customer_id  order_id
9         C06    1009.0


In [28]:
# C2 — match_status column
c2 = c1.copy()
def tag(row):
    if pd.notna(row['order_id']) and pd.notna(row['name']): return 'Both'
    elif pd.isna(row['order_id']): return 'Customer only'
    else: return 'Order only'
c2['match_status'] = c2.apply(tag, axis=1)
print("\nC2 — Match status counts:")
print(c2['match_status'].value_counts())



C2 — Match status counts:
match_status
Both             9
Order only       1
Customer only    1
Name: count, dtype: int64


In [29]:
# C3 — Full 3-table left join + revenue per city + flagging
c3 = pd.merge(orders, products, on='product_id', how='left')
c3 = pd.merge(c3, customers, on='customer_id', how='left')
c3['revenue'] = c3['quantity'] * c3['unit_price']
c3['city'] = c3['city'].fillna('Unknown')
city_rev = c3.groupby('city')['revenue'].sum().sort_values(ascending=False)
print("\nC3 — Revenue by city:")
print(city_rev)
top_city = city_rev.idxmax()
print(f"\nNumber: {top_city} generated ₹{city_rev.max():,.0f} | Reason: Multiple high-value orders from same city | Action: Prioritise {top_city} for next campaign.")
low_zones = city_rev[city_rev < 2000].index.tolist()
print(f"Low Revenue Zones (<₹2000): {low_zones}")



C3 — Revenue by city:
city
Mumbai       8050.0
Delhi        5600.0
Bangalore    4900.0
Chennai      1700.0
Hyderabad    1200.0
Unknown         0.0
Name: revenue, dtype: float64

Number: Mumbai generated ₹8,050 | Reason: Multiple high-value orders from same city | Action: Prioritise Mumbai for next campaign.
Low Revenue Zones (<₹2000): ['Chennai', 'Hyderabad', 'Unknown']


### D — Concat (20 pts)

**D1 (5 pts):** Stack `orders` (Q1) and `orders_q2` (Q2) vertically using `pd.concat()`.
Print the shape of the combined DataFrame and verify all order_ids are present.
Add a `quarter` column: `"Q1"` for the first 10 rows, `"Q2"` for the last 3.

**D2 (10 pts):** From the stacked orders, merge with `products` (inner join).
Add revenue column. 
Compare total revenue in Q1 vs Q2.
Print a trend observation using Number + Reason + Action.

**D3 — Bonus (5 pts):** After D1, use `pd.concat()` to add a summary row at the bottom showing:
- `order_id`: "TOTAL"
- `quantity`: total quantity
- `order_date`: "2024 Full Year"
Hint: Build a single-row DataFrame and concat it.


In [30]:
# D1 — Vertical concat + quarter column
combined = pd.concat([orders, orders_q2], ignore_index=True)
combined['quarter'] = ['Q1'] * len(orders) + ['Q2'] * len(orders_q2)
print(f"\nD1 — Combined shape: {combined.shape}")
print("All order_ids:", combined['order_id'].tolist())



D1 — Combined shape: (13, 6)
All order_ids: [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013]


In [31]:
# D2 — Stacked orders + products + Q1 vs Q2 revenue
d2 = pd.merge(combined, products, on='product_id', how='inner')
d2['revenue'] = d2['quantity'] * d2['unit_price']
q_rev = d2.groupby('quarter')['revenue'].sum()
print(f"\nD2 — Q1 revenue: ₹{q_rev.get('Q1',0):,.0f} | Q2 revenue: ₹{q_rev.get('Q2',0):,.0f}")
print(f"Number: Q2 revenue is ₹{q_rev.get('Q2',0):,.0f} vs Q1 ₹{q_rev.get('Q1',0):,.0f} | Reason: Only 3 Q2 orders vs 9 matched in Q1 | Action: Investigate if Q2 data is complete before drawing conclusions.")



D2 — Q1 revenue: ₹21,450 | Q2 revenue: ₹4,300
Number: Q2 revenue is ₹4,300 vs Q1 ₹21,450 | Reason: Only 3 Q2 orders vs 9 matched in Q1 | Action: Investigate if Q2 data is complete before drawing conclusions.


In [32]:
# D3 — Bonus: concat summary row
summary_row = pd.DataFrame([{
    'order_id':    'TOTAL',
    'customer_id': '',
    'product_id':  '',
    'quantity':    combined['quantity'].sum(),
    'order_date':  '2024 Full Year',
    'quarter':     ''
}])
d3 = pd.concat([combined, summary_row], ignore_index=True)
print(f"\nD3 — Final row:\n{d3.tail(1)}")



D3 — Final row:
   order_id customer_id product_id  quantity      order_date quarter
13    TOTAL                               24  2024 Full Year        


---
## 📊 Section 4 — Scoring Rubric

| Task | Points | What is checked |
|------|--------|-----------------|
| A1 | 5 | Correct inner join, shape printed, comparison line showing lost rows |
| A2 | 5 | Correct join, revenue column computed, total printed |
| A3 | 10 | 3-table merge correct, top customer by name, top category, N+R+A format |
| B1 | 5 | Left join, NaN orders identified, comment explaining C06 |
| B2 | 5 | NaN filled correctly, value_counts with "Unregistered" segment |
| B3 | 10 | Unmatched orders found, revenue split, missing revenue estimated, N+R+A |
| C1 | 5 | Outer join, customers with no orders AND orders with no customer both found |
| C2 | 5 | match_status column with 3 categories, value_counts printed |
| C3 | 10 | Full 3-table left merge, revenue by city, top city insight, low zone flag |
| D1 | 5 | Vertical concat, shape correct, quarter column added |
| D2 | 10 | Revenue by quarter, Q1 vs Q2 comparison, N+R+A insight |
| D3 | 5 ★ | Summary row appended correctly using concat |
| **Total** | **80 + 5★** | |

**Automatic deductions:**
- Silent cell (no print output): −2 per task
- Hardcoded value instead of computed: −2
- Missing N+R+A where required: −3
- Wrong join type used: −5 (this is the key skill of the day)


---
## 🎙️ Interview Angle

**Q: "Walk me through how you join data from multiple sources in Python. What can go wrong?"**

**A (model answer):**
"I use `pd.merge()` with explicit `how=` parameter — inner, left, right, or outer depending on the business question.
The most common mistake is defaulting to inner join when you need a left join — you silently lose rows.
I always check row counts before and after merging, and look for unexpected NaN columns in the result.
If two tables have a column with the same name but different meaning, I handle the suffix conflict before or during the merge.
For stacking same-schema data (like Q1 + Q2), I use `pd.concat()` instead, which is the pandas equivalent of SQL UNION ALL."

---

**Key Takeaway — Day 39:**
A merge is only as good as your understanding of what rows you *expect* to lose.
Inner join silently drops unmatched rows. Left join is usually safer for analysis — it preserves your base table.
Always check `before_count vs after_count` after any merge. Silent data loss is the most dangerous bug in client work.
